# Notebook 3 — Perbandingan Metode (Proyek FERQUEUE vs PCV Klasik Adaptif NB1 vs Metode yang Tidak Digunakan)

Notebook ini adalah **studi perbandingan ilmiah** yang menjalankan seluruh metode pada **gambar uji yang SAMA** (misal 4 gambar `data/helper`) untuk menjawab pertanyaan kunci saat presentasi:
> *"Mengapa proyek FERQUEUE memilih SSD MobileNet V2 FPNLite + Baseline RGB (+ GrabCut pasca-deteksi), bukan PCV klasik adaptif (Notebook 1) atau metode/arsitektur lainnya?"*

### 3 Kelompok Metode yang Dibandingkan (100% Sinkron dengan Notebook 1 & 2)
1. **Metode Proyek FERQUEUE (Notebook 2 — Digunakan di Produksi)**:
   - Baseline RGB + Resize `320×320` + Normalisasi `[-1, 1]` → **SSD MobileNet V2 FPNLite** (Ekstraksi fitur otomatis P3–P7 + NMS) → Bounding Box + Label (`Mobil`/`Truk`/`Bus`) + Confidence (`~25–45 ms/gambar`) + *(opsional GrabCut pasca-deteksi)*.
2. **Metode PCV Klasik Adaptif (Notebook 1 — Terkalibrasi pada 1.079 Gambar `data/inference`)**:
   - Auto-Letterbox → **Saliency Klasik (CIELAB + FFT) + GrabCut `if-else`** (`margin=0.05`, `silau > 22%`, `LAB-CLAHE + Gamma 0.78` saat gelap `< 85`) → Ekstraksi Fitur Geometri/HOG/GLCM → Bounding Box manual tanpa label (`Mean IoU 0.73`, `87.7% IoU≥0.5`, tetapi lambat `~980 ms/gambar` & lemah di silau ekstrem `37.2%`).
3. **Metode yang TIDAK Digunakan (Diuji Langsung Secara Visual + Matriks Arsitektur)**:
   - **Preprocessing non-baseline**: Gaussian Blur + CLAHE + Gamma pada input SSD.
   - **Segmentasi klasik dasar**: **Otsu Global**, **K-Means Color Clustering**, **Watershed**, **Canny+Morfologi Naive** *(bocor full-frame 57.7%)*, dan **HSV Warna**.
   - **Detektor & Segmentasi lain**: **HOG + SVM / Haar Cascade**, **Faster R-CNN / SSD ResNet50**, dan **Mask R-CNN / U-Net**.

## Cell 1 — Setup Library & Folder Output Presentasi

In [ ]:
import json
import math
import shutil
import zipfile
from pathlib import Path
from time import perf_counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from skimage.feature import hog
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display  # type: ignore[name-defined]
except NameError:
    display = print

OUT_DIR = Path("/content/outputs_presentasi/nb3_perbandingan") if IN_COLAB else Path("./outputs_presentasi/nb3_perbandingan")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_WORK_DIR = Path("/content/ferqueue_model_viz") if IN_COLAB else Path("./outputs_presentasi/_model_cache")
MODEL_WORK_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 10.5
plt.rcParams["axes.titleweight"] = "bold"


def save_fig(fig, filename: str):
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Tersimpan: {path}")
    return path


print(f"Siap! TensorFlow={tf.__version__} | OpenCV={cv2.__version__} | Output={OUT_DIR}")

## Cell 2 — Muat Model FERQUEUE (`ckpt-35`) & 4 Gambar Uji (`data/helper`)

* Unggah file model `pcv-model-colab-v3-ckpt-35.zip` (jika sebelumnya sudah menjalankan Notebook 2 di sesi Colab yang sama, model otomatis dipakai ulang dari cache tanpa perlu unggah 2×).
* Unggah 4 gambar dari folder `data/helper` agar 100% konsisten dengan demo Notebook 1 & Notebook 2.

In [ ]:
MODEL_SOURCE = "upload_zip"  # "upload_zip" | "drive" | "local" (otomatis reuse jika sudah diekstrak dari NB2)
DRIVE_MODEL_PATH = "/content/drive/MyDrive/ferqueue/pcv-coco-colab-v3/pcv-model-colab-v3-ckpt-35.zip"
LOCAL_MODEL_PATH = "models/mobilenet/runs/pcv-receiver-v3-unfrozen/exports/imported-ckpt-35"

IMAGE_SOURCE = "upload"      # "upload" | "drive" | "local"
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/ferqueue/helper"
LOCAL_IMAGE_DIR = "data/helper"
ACTIVE_INDEX = 0             # Indeks gambar utama untuk Eksperimen 1-3 (Eksperimen 4 jalankan semua gambar)

CLASSES = ["Mobil", "Truk", "Bus"]
CLASS_COLORS = {"Mobil": (40, 210, 90), "Truk": (255, 165, 0), "Bus": (65, 155, 255)}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# 1. Muat SavedModel (cek cache NB2 terlebih dahulu agar tidak perlu upload 2x di sesi Colab yang sama)
cached_saved = MODEL_WORK_DIR / "extracted/saved_model"
if (cached_saved / "saved_model.pb").is_file():
    saved_model_dir = cached_saved
    if (MODEL_WORK_DIR / "extracted/model.json").is_file():
        CLASSES = json.loads((MODEL_WORK_DIR / "extracted/model.json").read_text(encoding="utf-8")).get("classes", CLASSES)
    print(f"Menggunakan SavedModel dari cache sesi Colab: {saved_model_dir}")
else:
    if MODEL_SOURCE == "upload_zip" and IN_COLAB:
        print("Silakan unggah ZIP model FERQUEUE (pcv-model-colab-v3-ckpt-35.zip)...")
        up = colab_files.upload()
        zname, zbytes = next(iter(up.items()))
        zpath = MODEL_WORK_DIR / Path(zname).name
        zpath.write_bytes(zbytes)
        target = zpath
    elif MODEL_SOURCE == "drive" and IN_COLAB:
        drive.mount("/content/drive")
        target = Path(DRIVE_MODEL_PATH)
    else:
        target = Path(LOCAL_MODEL_PATH)
        if not target.exists() and Path("outputs/pcv-model-colab-v3-ckpt-35.zip").is_file():
            target = Path("outputs/pcv-model-colab-v3-ckpt-35.zip")

    if target.is_file() and target.suffix.lower() == ".zip":
        ext = MODEL_WORK_DIR / "extracted"
        if ext.exists():
            shutil.rmtree(ext)
        ext.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(target) as zf:
            zf.extractall(ext)
        if (ext / "model.json").is_file():
            CLASSES = json.loads((ext / "model.json").read_text(encoding="utf-8")).get("classes", CLASSES)
        saved_model_dir = ext / "saved_model"
    else:
        saved_model_dir = target / "saved_model" if (target / "saved_model/saved_model.pb").is_file() else target

loaded_tf_model = tf.saved_model.load(str(saved_model_dir))
detector_fn = loaded_tf_model.signatures["serving_default"]

# Siapkan ekstraktor feature map FPN (P3-P5) untuk perbandingan ekstraksi fitur
frozen_fn = convert_variables_to_constants_v2(detector_fn, lower_control_flow=False)
all_tensors = [o.name for op in frozen_fn.graph.get_operations() for o in op.outputs]
fpn_suffixes = [
    "PredictionTower/conv2d_3/activation_0/Relu6:0",
    "PredictionTower/conv2d_3/activation_1/Relu6:0",
    "PredictionTower/conv2d_3/activation_2/Relu6:0",
]
fpn_fetches = [
    frozen_fn.graph.get_tensor_by_name(next(n for n in all_tensors if n.endswith(s)))
    for s in fpn_suffixes
    if any(n.endswith(s) for n in all_tensors)
]
fpn_extractor = frozen_fn.prune(frozen_fn.inputs, fpn_fetches)

# 2. Muat gambar uji
UPLOAD_DIR = OUT_DIR / "_inputs"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
image_paths = []
if IMAGE_SOURCE == "upload" and IN_COLAB:
    print("\nSilakan unggah 4 gambar uji dari folder data/helper...")
    up_imgs = colab_files.upload()
    for fname, content in up_imgs.items():
        dst = UPLOAD_DIR / Path(fname).name
        dst.write_bytes(content)
        if dst.suffix.lower() in IMAGE_EXTS:
            image_paths.append(dst)
elif IMAGE_SOURCE == "drive" and IN_COLAB:
    drive.mount("/content/drive")
    image_paths = sorted(p for p in Path(DRIVE_IMAGE_DIR).iterdir() if p.suffix.lower() in IMAGE_EXTS)
else:
    if Path(LOCAL_IMAGE_DIR).is_dir():
        image_paths = sorted(p for p in Path(LOCAL_IMAGE_DIR).iterdir() if p.suffix.lower() in IMAGE_EXTS)

if not image_paths:
    raise FileNotFoundError("Belum ada gambar uji yang dimuat.")


def load_rgb(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        return np.asarray(ImageOps.exif_transpose(img).convert("RGB"), dtype=np.uint8)


loaded_images = [(p.name, load_rgb(p)) for p in image_paths]
ACTIVE_INDEX = min(ACTIVE_INDEX, len(loaded_images) - 1)
active_name, img_rgb = loaded_images[ACTIVE_INDEX]

# Warmup model 1x
_ = detector_fn(tf.convert_to_tensor(img_rgb[None, ...], dtype=tf.uint8))

print(f"Siap! Model dimuat ({CLASSES}) | {len(loaded_images)} gambar uji | Aktif=[{ACTIVE_INDEX}] {active_name}")

## Cell 3 — Fungsi Inti Semua Metode (Sinkron dengan Notebook 1 & Notebook 2)

* **FERQUEUE (Digunakan — NB2)**: `run_ssd_detect`, `grabcut_from_boxes`, FPNLite heatmap.
* **PCV Klasik Adaptif (Pembanding Utama — NB1 terkalibrasi 1.079 gambar)**: `detect_black_letterbox`, `compute_classical_saliency`, `classic_nb1_detect` (`margin=0.05`, `silau > 22%`, `dark LAB-CLAHE + Gamma 0.78`, `peak_fallback=True`).
* **Metode yang TIDAK Digunakan**: `apply_extra_preprocessing`, `segment_otsu`, `segment_kmeans`, `segment_watershed`, `segment_canny_naive`, `segment_hsv`.

In [ ]:
CONF_TH = 0.50
NMS_IOU = 0.45


def apply_extra_preprocessing(rgb: np.ndarray, denoise_k=5, clahe_clip=2.5, gamma=1.0) -> np.ndarray:
    out = rgb.copy()
    if denoise_k:
        out = cv2.GaussianBlur(out, (denoise_k, denoise_k), 0)
    if clahe_clip:
        lab = cv2.cvtColor(out, cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8)).apply(lab[:, :, 0])
        out = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    if gamma != 1.0:
        lut = np.round(255.0 * (np.arange(256) / 255.0) ** gamma).astype(np.uint8)
        out = cv2.LUT(out, lut)
    return out


def compute_iou_xyxy(a, b) -> float:
    if a is None or b is None:
        return 0.0
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - inter
    return float(inter / union) if union > 0 else 0.0


def select_detections(boxes, scores, classes, conf_th=CONF_TH, iou_th=NMS_IOU, max_det=10):
    candidates = [
        int(i) for i in np.argsort(-scores, kind="stable")
        if scores[i] >= conf_th and boxes[i][2] > boxes[i][0] and boxes[i][3] > boxes[i][1]
    ]
    keep = []
    while candidates and len(keep) < max_det:
        cur = candidates.pop(0)
        keep.append(cur)
        a = boxes[cur]
        rem = []
        for i in candidates:
            b = boxes[i]
            inter = max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
            union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
            if union <= 0 or inter / union <= iou_th:
                rem.append(i)
        candidates = rem
    return keep


def run_ssd_detect(rgb: np.ndarray, conf_th=CONF_TH):
    t0 = perf_counter()
    res = detector_fn(tf.convert_to_tensor(rgb[None, ...], dtype=tf.uint8))
    ms = (perf_counter() - t0) * 1000.0
    n = int(res["num_detections"][0])
    boxes = res["detection_boxes"][0, :n].numpy()
    scores = res["detection_scores"][0, :n].numpy()
    classes = res["detection_classes"][0, :n].numpy()
    keep = select_detections(boxes, scores, classes, conf_th=conf_th)
    h, w = rgb.shape[:2]
    dets = []
    for k in keep:
        cid = int(classes[k])
        cname = CLASSES[cid - 1] if 1 <= cid <= len(CLASSES) else f"ID-{cid}"
        y1, x1, y2, x2 = boxes[k].tolist()
        dets.append({
            "class": cname,
            "score": float(scores[k]),
            "box": [y1, x1, y2, x2],
            "xyxy_px": [x1 * w, y1 * h, x2 * w, y2 * h],
        })
    return dets, ms


def draw_ssd_boxes(rgb: np.ndarray, dets: list) -> np.ndarray:
    vis = rgb.copy()
    h, w = vis.shape[:2]
    for d in dets:
        y1, x1, y2, x2 = d["box"]
        col = CLASS_COLORS.get(d["class"], (40, 210, 90))
        px1, py1, px2, py2 = int(x1 * w), int(y1 * h), int(x2 * w), int(y2 * h)
        cv2.rectangle(vis, (px1, py1), (px2, py2), col, 3)
        txt = f"{d['class']} {d['score']:.2f}"
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
        cv2.rectangle(vis, (px1, max(0, py1 - th - 8)), (px1 + tw + 6, max(th + 8, py1)), col, -1)
        cv2.putText(vis, txt, (px1 + 3, max(th + 3, py1 - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 2, cv2.LINE_AA)
    return vis


def grabcut_from_boxes(rgb: np.ndarray, dets: list) -> np.ndarray:
    h, w = rgb.shape[:2]
    combined = np.zeros((h, w), dtype=np.uint8)
    for d in dets:
        y1n, x1n, y2n, x2n = d["box"]
        x1, y1 = max(0, math.floor(x1n * w)), max(0, math.floor(y1n * h))
        x2, y2 = min(w, math.ceil(x2n * w)), min(h, math.ceil(y2n * h))
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        padded = cv2.copyMakeBorder(rgb, 1, 1, 1, 1, cv2.BORDER_REPLICATE)
        mask = np.zeros(padded.shape[:2], dtype=np.uint8)
        cv2.setRNGSeed(42)
        try:
            cv2.grabCut(padded, mask, (x1 + 1, y1 + 1, x2 - x1, y2 - y1),
                        np.zeros((1, 65)), np.zeros((1, 65)), 5, cv2.GC_INIT_WITH_RECT)
        except cv2.error:
            continue
        fg = np.isin(mask[1:-1, 1:-1], (cv2.GC_FGD, cv2.GC_PR_FGD)).astype(np.uint8) * 255
        elem = np.ones((3, 3), dtype=np.uint8)
        fg = cv2.morphologyEx(cv2.morphologyEx(fg, cv2.MORPH_OPEN, elem), cv2.MORPH_CLOSE, elem)
        bounded = np.zeros_like(fg)
        bounded[y1:y2, x1:x2] = fg[y1:y2, x1:x2]
        combined = cv2.bitwise_or(combined, bounded)
    return combined


# === PCV KLASIK ADAPTIF (IDENTIK DENGAN NOTEBOOK 1 TERKALIBRASI 1.079 GAMBAR) ===
def detect_black_letterbox(rgb: np.ndarray) -> tuple:
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    bad_r = (gray.mean(axis=1) < 14) & (gray.std(axis=1) < 10)
    bad_c = (gray.mean(axis=0) < 14) & (gray.std(axis=0) < 10)
    y1, y2, x1, x2 = 0, h, 0, w
    while y1 < h // 3 and bad_r[y1]:
        y1 += 1
    while y2 > 2 * h // 3 and bad_r[y2 - 1]:
        y2 -= 1
    while x1 < w // 3 and bad_c[x1]:
        x1 += 1
    while x2 > 2 * w // 3 and bad_c[x2 - 1]:
        x2 -= 1
    return y1, y2, x1, x2


def compute_classical_saliency(crop: np.ndarray, k_bg: int = 6) -> np.ndarray:
    h, w = crop.shape[:2]
    blur = cv2.GaussianBlur(crop, (7, 7), 0)
    lab = cv2.cvtColor(blur, cv2.COLOR_RGB2LAB).astype(np.float32)
    by, bx = max(4, int(0.08 * h)), max(4, int(0.08 * w))
    border_pts = np.vstack([
        lab[:by, :, :].reshape(-1, 3),
        lab[h - by :, :, :].reshape(-1, 3),
        lab[:, :bx, :].reshape(-1, 3),
        lab[:, w - bx :, :].reshape(-1, 3),
    ])
    step = max(1, len(border_pts) // 2500)
    sample = border_pts[::step].astype(np.float32)
    cv2.setRNGSeed(42)
    _, _, centers = cv2.kmeans(
        sample, k_bg, None,
        (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0),
        3, cv2.KMEANS_PP_CENTERS,
    )
    dists = np.linalg.norm(lab.reshape(-1, 3)[:, None, :] - centers[None, :, :], axis=2)
    sal_lab = cv2.normalize(dists.min(axis=1).reshape(h, w), None, 0.0, 1.0, cv2.NORM_MINMAX)

    small = cv2.resize(cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY), (64, 64)).astype(np.float32)
    fft = np.fft.fft2(small)
    log_amp = np.log(np.abs(fft) + 1e-8)
    residual = log_amp - cv2.blur(log_amp, (3, 3))
    sal_s = np.abs(np.fft.ifft2(np.exp(residual + 1j * np.angle(fft)))) ** 2
    sal_s = cv2.GaussianBlur(sal_s.astype(np.float32), (9, 9), 2.5)
    sal_spec = cv2.normalize(cv2.resize(sal_s, (w, h), interpolation=cv2.INTER_CUBIC), None, 0.0, 1.0, cv2.NORM_MINMAX)

    yy, xx = np.mgrid[0:h, 0:w]
    dx = (xx - w * 0.5) / (w * 0.5)
    dy = (yy - h * 0.5) / (h * 0.5)
    center_w = np.exp(-0.75 * (dx ** 2 + dy ** 2))
    comb = (0.60 * sal_lab + 0.40 * sal_spec) * (0.30 + 0.70 * center_w)
    sal_u8 = cv2.normalize(comb, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return cv2.GaussianBlur(sal_u8, (9, 9), 0)


def classic_nb1_detect(rgb: np.ndarray):
    """Pipeline PCV Klasik Adaptif persis seperti Notebook 1 (terkalibrasi 1.079 gambar)."""
    t0 = perf_counter()
    h0, w0 = rgb.shape[:2]
    y1, y2, x1, x2 = detect_black_letterbox(rgb)
    crop = rgb[y1:y2, x1:x2]
    h, w = crop.shape[:2]
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    mean_g = float(gray.mean())
    bright_r = float(np.mean(gray > 230))

    work_crop = crop
    if mean_g < 85.0:
        lab = cv2.cvtColor(work_crop, cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = cv2.createCLAHE(clipLimit=2.2, tileGridSize=(8, 8)).apply(lab[:, :, 0])
        work_crop = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        lut = np.round(255.0 * (np.arange(256) / 255.0) ** 0.78).astype(np.uint8)
        work_crop = cv2.LUT(work_crop, lut)

    sal = compute_classical_saliency(work_crop, k_bg=6)
    scale = min(1.0, 320.0 / max(h, w))
    sw, sh = max(32, int(w * scale)), max(32, int(h * scale))
    crop_s = cv2.resize(work_crop, (sw, sh), interpolation=cv2.INTER_AREA)
    sal_s = cv2.resize(sal, (sw, sh), interpolation=cv2.INTER_AREA)
    gray_s = cv2.cvtColor(crop_s, cv2.COLOR_RGB2GRAY)

    cv2.setRNGSeed(42)
    bgd = np.zeros((1, 65), np.float64)
    fgd = np.zeros((1, 65), np.float64)
    strategy = "inner_rect"
    used_peak = False

    if bright_r > 0.22:
        sal_inner = sal_s.copy()
        bx, by = max(4, int(0.08 * sw)), max(4, int(0.08 * sh))
        sal_inner[:, :bx] = 0
        sal_inner[:, sw - bx :] = 0
        sal_inner[:by, :] = 0
        sal_inner[sh - by :, :] = 0
        peak_th = max(120, int(0.68 * sal_inner.max()))
        core = (sal_inner >= peak_th).astype(np.uint8) * 255
        core = cv2.morphologyEx(core, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
        cnts_c, _ = cv2.findContours(core, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cnts_c:
            def peak_sc(c):
                m = np.zeros((sh, sw), dtype=np.uint8)
                cv2.drawContours(m, [c], -1, 255, cv2.FILLED)
                vals = sal_s[m > 0]
                return (float(vals.max()), float(vals.mean()))
            best_core = max(cnts_c, key=peak_sc)
            cx, cy, cw, ch = cv2.boundingRect(best_core)
            est_w = max(cw, int(0.32 * sw))
            mid_x = cx + cw // 2
            rx1 = max(2, int(mid_x - 0.85 * est_w))
            rx2 = min(sw - 3, int(mid_x + 0.65 * est_w))
            non_sky_frac = np.mean(gray_s[:, rx1:rx2] < 232, axis=1)
            ry1, streak = cy, 0
            for yy_idx in range(cy, 1, -1):
                if non_sky_frac[yy_idx] > 0.08:
                    ry1, streak = yy_idx, 0
                else:
                    streak += 1
                    if streak >= max(8, int(0.04 * sh)):
                        break
            ry1 = max(2, ry1 - int(0.04 * sh))
            ry2 = min(sh - 3, cy + ch + int(0.08 * sh))
            gc = np.full((sh, sw), cv2.GC_BGD, dtype=np.uint8)
            gc[ry1:ry2, rx1:rx2] = cv2.GC_PR_FGD
            gc[gray_s > 235] = cv2.GC_BGD
            core_fg = np.zeros((sh, sw), dtype=np.uint8)
            cv2.drawContours(core_fg, [best_core], -1, 255, cv2.FILLED)
            gc[(core_fg > 0) & (gc != cv2.GC_BGD)] = cv2.GC_FGD
            cv2.grabCut(crop_s, gc, None, bgd, fgd, 5, cv2.GC_INIT_WITH_MASK)
            fg_raw = np.isin(gc, (cv2.GC_FGD, cv2.GC_PR_FGD)).astype(np.uint8) * 255
            fg_s = cv2.morphologyEx(fg_raw, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 25)))
            used_peak = True
            strategy = "peak_saliency_roi"
            if np.mean(fg_s > 0) < 0.03:
                used_peak = False
                strategy = "inner_rect"

    if not used_peak:
        mx, my = max(2, int(0.05 * sw)), max(2, int(0.05 * sh))
        gc = np.zeros((sh, sw), dtype=np.uint8)
        cv2.grabCut(crop_s, gc, (mx, my, sw - 2 * mx, sh - 2 * my), bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)
        fg_s = np.isin(gc, (cv2.GC_FGD, cv2.GC_PR_FGD)).astype(np.uint8) * 255

    fg_s = cv2.morphologyEx(fg_s, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)))
    fg_s = cv2.morphologyEx(fg_s, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    fg_crop = cv2.resize(fg_s, (w, h), interpolation=cv2.INTER_NEAREST)
    cnts, _ = cv2.findContours(fg_crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best_crop = np.zeros_like(fg_crop)
    if cnts:
        cv2.drawContours(best_crop, [max(cnts, key=cv2.contourArea)], -1, 255, cv2.FILLED)

    full_mask = np.zeros((h0, w0), dtype=np.uint8)
    full_mask[y1:y2, x1:x2] = best_crop
    full_sal = np.zeros((h0, w0), dtype=np.uint8)
    full_sal[y1:y2, x1:x2] = sal

    ms = (perf_counter() - t0) * 1000.0
    vis = rgb.copy()
    bbox_xyxy = None
    cnts_f, _ = cv2.findContours(full_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts_f:
        x, y, bw, bh = cv2.boundingRect(max(cnts_f, key=cv2.contourArea))
        cv2.rectangle(vis, (x, y), (x + bw, y + bh), (40, 220, 80), 4)
        bbox_xyxy = [float(x), float(y), float(x + bw), float(y + bh)]
    return vis, full_mask, full_sal, bbox_xyxy, strategy, ms


# === METODE SEGMENTASI KLASIK DASAR YANG TIDAK DIPAKAI ===
def auto_orient(mask: np.ndarray) -> np.ndarray:
    border = np.concatenate([mask[0, :], mask[-1, :], mask[:, 0], mask[:, -1]])
    return cv2.bitwise_not(mask) if np.mean(border > 0) > 0.5 else mask


def clean_and_fill(mask: np.ndarray, open_k=5, close_k=13) -> np.ndarray:
    ko = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k))
    kc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
    m = cv2.morphologyEx(cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kc), cv2.MORPH_OPEN, ko)
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = np.zeros_like(m)
    min_a = 0.015 * m.shape[0] * m.shape[1]
    for c in cnts:
        if cv2.contourArea(c) >= min_a:
            cv2.drawContours(out, [c], -1, 255, cv2.FILLED)
    return out


def segment_otsu(rgb: np.ndarray) -> np.ndarray:
    gray = cv2.GaussianBlur(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    _, raw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return clean_and_fill(auto_orient(raw))


def segment_kmeans(rgb: np.ndarray, k=3) -> np.ndarray:
    small = cv2.resize(rgb, (160, 160), interpolation=cv2.INTER_AREA)
    data = small.reshape((-1, 3)).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 15, 1.0)
    cv2.setRNGSeed(42)
    _, labels, _ = cv2.kmeans(data, k, None, criteria, 3, cv2.KMEANS_PP_CENTERS)
    label_img = labels.reshape(small.shape[:2])
    border_labels = np.concatenate([label_img[0, :], label_img[-1, :], label_img[:, 0], label_img[:, -1]])
    bg_cluster = int(np.bincount(border_labels).argmax())
    fg_small = (label_img != bg_cluster).astype(np.uint8) * 255
    fg = cv2.resize(fg_small, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
    return clean_and_fill(fg)


def segment_watershed(rgb: np.ndarray) -> np.ndarray:
    gray = cv2.GaussianBlur(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresh = auto_orient(thresh)
    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, 0.35 * dist.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    unknown = cv2.subtract(sure_bg, sure_fg)
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    ws_markers = cv2.watershed(rgb.copy(), markers)
    fg = (ws_markers > 1).astype(np.uint8) * 255
    return clean_and_fill(fg)


def segment_canny_naive(rgb: np.ndarray) -> np.ndarray:
    gray = cv2.GaussianBlur(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    enh = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    edges = cv2.Canny(enh, 50, 140)
    dil = cv2.dilate(edges, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=2)
    return clean_and_fill(dil)


def make_overlay(rgb: np.ndarray, mask: np.ndarray, color=(255, 70, 20), alpha=0.45) -> np.ndarray:
    ov = rgb.copy().astype(np.float32)
    tint = np.zeros_like(ov)
    tint[:, :] = color
    m = mask > 0
    ov[m] = (1 - alpha) * ov[m] + alpha * tint[m]
    return ov.astype(np.uint8)


print("Semua fungsi metode pembanding (sinkron NB1 & NB2) siap!")

## Cell 4 — Eksperimen 1: Perbandingan Preprocessing (Baseline RGB FERQUEUE vs Filter Eksternal yang Tidak Diaktifkan)

Di proyek FERQUEUE (`receiver-v3.json`), model final dilatih dengan **Baseline RGB** (`denoise=0, clahe=0, gamma=1.0`), sedangkan Gaussian Blur, CLAHE, dan Gamma disediakan sebagai opsi yang **tidak diaktifkan** pada model final.
Mari bandingkan langsung efek ketiga kondisi preprocessing terhadap prediksi SSD MobileNetV2!

In [ ]:
prep_variants = [
    ("1. Baseline RGB\n(DIPAKAI di FERQUEUE v3)", img_rgb),
    ("2. Gaussian(5)+CLAHE(2.5)\n(TIDAK diaktifkan di v3)", apply_extra_preprocessing(img_rgb, 5, 2.5, 1.0)),
    ("3. Gaussian(5)+CLAHE+Gamma(1.4)\n(TIDAK diaktifkan di v3)", apply_extra_preprocessing(img_rgb, 5, 2.5, 1.4)),
]

fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.2))
prep_rows = []
for col, (title, im_var) in enumerate(prep_variants):
    dets_v, ms_v = run_ssd_detect(im_var)
    vis_v = draw_ssd_boxes(im_var, dets_v)
    top_str = ", ".join(f"{d['class']} ({d['score']:.2f})" for d in dets_v) or "Tidak terdeteksi"
    prep_rows.append({
        "Varian_Preprocessing": title.replace("\n", " "),
        "Jumlah_Deteksi": len(dets_v),
        "Hasil_Prediksi": top_str,
        "Max_Confidence": round(max((d["score"] for d in dets_v), default=0.0), 4),
        "Waktu_Inferensi_ms": round(ms_v, 1),
    })
    axes[0, col].imshow(im_var)
    axes[0, col].set_title(title)
    axes[1, col].imshow(vis_v)
    axes[1, col].set_title(f"Prediksi: {top_str}\n({ms_v:.1f} ms)")
    axes[0, col].axis("off")
    axes[1, col].axis("off")

fig.suptitle(f"Eksperimen 1: Pengaruh Preprocessing terhadap Prediksi SSD MobileNetV2 — {active_name[:34]}", fontsize=13.5, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.94])
save_fig(fig, "01_banding_preprocessing.png")
plt.show()

display(pd.DataFrame(prep_rows))

## Cell 5 — Eksperimen 2: Perbandingan 6 Metode Segmentasi (Metode Tidak Dipakai vs PCV Klasik NB1 vs GrabCut Dipandu SSD)

Perbandingan visual langsung pada gambar yang sama:
1. **Otsu Global** *(Tidak Dipakai — sensitif kontras terang-gelap)*
2. **K-Means K=3** *(Tidak Dipakai — memecah bodi/kaca/roda jadi klaster terpisah)*
3. **Watershed** *(Tidak Dipakai — rentan over-segmentation pada tekstur latar)*
4. **Canny + Morfologi Naive** *(Tidak Dipakai — bocor full-frame 57.7% di 1.079 gambar)*
5. **Saliency + GrabCut Adaptif (`if-else`)** *(PCV Klasik NB1 — 87.7% IoU≥0.5 di 1.079 gambar, tetapi ~980 ms)*
6. **GrabCut + Bbox SSD FERQUEUE** *(Metode Review NB2 — dibatasi langsung oleh deteksi AI)*

In [ ]:
ssd_dets_active, _ = run_ssd_detect(img_rgb)

seg_benchmark = []
seg_specs = [
    ("Otsu Global\n(Tidak Dipakai)", lambda im: segment_otsu(im), (255, 165, 0)),
    ("K-Means K=3\n(Tidak Dipakai)", lambda im: segment_kmeans(im, 3), (180, 90, 255)),
    ("Watershed\n(Tidak Dipakai)", lambda im: segment_watershed(im), (60, 170, 255)),
    ("Canny Naive\n(Tidak Dipakai)", lambda im: segment_canny_naive(im), (220, 80, 80)),
    ("Saliency+GrabCut\n(PCV Klasik NB1)", lambda im: classic_nb1_detect(im)[1], (40, 220, 90)),
    ("GrabCut+Bbox SSD\n(FERQUEUE NB2)", lambda im: grabcut_from_boxes(im, ssd_dets_active), (255, 70, 20)),
]

fig, axes = plt.subplots(2, 6, figsize=(22, 7.6))
for col, (label, fn, ov_color) in enumerate(seg_specs):
    t0 = perf_counter()
    mask_m = fn(img_rgb)
    ms_m = (perf_counter() - t0) * 1000.0
    cov = 100.0 * np.mean(mask_m > 0)
    seg_benchmark.append({
        "Metode_Segmentasi": label.replace("\n", " "),
        "Cakupan_Foreground_%": round(cov, 1),
        "Waktu_Proses_ms": round(ms_m, 1),
    })
    axes[0, col].imshow(mask_m, cmap="gray")
    axes[0, col].set_title(f"{label}\nArea={cov:.1f}% | {ms_m:.0f}ms", fontsize=9)
    axes[1, col].imshow(make_overlay(img_rgb, mask_m, color=ov_color))
    axes[1, col].set_title("Overlay Foreground", fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].axis("off")

fig.suptitle(f"Eksperimen 2: Perbandingan 6 Metode Segmentasi — {active_name[:34]}", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.94])
save_fig(fig, "02_banding_6_segmentasi.png")
plt.show()

display(pd.DataFrame(seg_benchmark))

## Cell 6 — Eksperimen 3: Perbandingan Paradigma Ekstraksi Fitur (HSV Warna vs Saliency/HOG NB1 vs MobileNetV2+FPNLite NB2)

Membandingkan bagaimana 4 representasi fitur "melihat" kendaraan yang sama:
1. **Fitur Warna HSV** *(Tidak Dipakai)* — gagal membedakan kendaraan warna netral (putih/silver/hitam) dari aspal.
2. **Peta Saliency Klasik (CIELAB + FFT — NB1)** — menonjolkan area berbeda dari pinggir gambar tanpa training.
3. **Fitur Gradien Manual (HOG — NB1)** — menangkap pola orientasi tepi secara *hand-crafted*.
4. **Activation Heatmap MobileNetV2 + FPNLite (FERQUEUE — NB2)** — fitur CNN yang dipelajari otomatis dari 904 gambar latih sehingga fokus tepat ke bentuk `Mobil`/`Truk`/`Bus`.

In [ ]:
hsv_map = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)[:, :, 1]
_, _, sal_active, _, _, _ = classic_nb1_detect(img_rgb)

gray_256 = cv2.resize(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY), (256, 256))
hog_vec_cmp, hog_vis_cmp = hog(
    gray_256, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2),
    block_norm="L2-Hys", visualize=True,
)

H, W = img_rgb.shape[:2]
fpn_outs = fpn_extractor(tf.convert_to_tensor(img_rgb[None, ...], dtype=tf.uint8))
hm = np.zeros((H, W), dtype=np.float32)
for t_out in fpn_outs:
    m = t_out.numpy()[0].mean(axis=-1)
    m = (m - m.min()) / (m.max() - m.min() + 1e-6)
    hm += cv2.resize(m, (W, H), interpolation=cv2.INTER_CUBIC)
hm = (hm - hm.min()) / (hm.max() - hm.min() + 1e-6)
hm_rgb = cv2.cvtColor(cv2.applyColorMap((hm * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
hm_overlay = cv2.addWeighted(img_rgb, 0.55, hm_rgb, 0.45, 0)

fig, axes = plt.subplots(1, 4, figsize=(17.5, 4.5))
axes[0].imshow(hsv_map, cmap="plasma")
axes[0].set_title("1. Fitur Warna HSV (Tidak Dipakai)\nGagal di warna putih/silver/hitam")
axes[1].imshow(sal_active, cmap="inferno")
axes[1].set_title("2. Saliency Lab+FFT (PCV Klasik NB1)\nKontras piksel tanpa training")
axes[2].imshow(hog_vis_cmp, cmap="inferno")
axes[2].set_title(f"3. Fitur HOG (PCV Klasik NB1)\nManual ({hog_vec_cmp.size} dimensi)")
axes[3].imshow(hm_overlay)
axes[3].set_title("4. MobileNetV2+FPNLite (FERQUEUE)\nLearned otomatis fokus ke kendaraan")
for ax in axes:
    ax.axis("off")

fig.suptitle(f"Eksperimen 3: Perbandingan Paradigma Ekstraksi Fitur — {active_name[:34]}", fontsize=13.5, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.92])
save_fig(fig, "03_banding_ekstraksi_fitur.png")
plt.show()

## Cell 7 — Eksperimen 4 (Puncak Presentasi): Perbandingan Hasil Akhir Side-by-Side (PCV Klasik NB1 vs FERQUEUE SSD MobileNetV2)

Menjalankan **PCV Klasik Adaptif (Notebook 1 — Saliency + GrabCut `if-else`, BBox Tanpa Label)** berdampingan dengan **FERQUEUE SSD MobileNetV2 (Notebook 2 — BBox + Label Kelas + Confidence)** pada semua gambar uji, sekaligus menghitung **IoU antar keduanya** dan membandingkan **kecepatan eksekusi (`ms`)**!

In [ ]:
n_imgs = len(loaded_images)
fig, axes = plt.subplots(n_imgs, 4, figsize=(17.5, 4.0 * n_imgs))
if n_imgs == 1:
    axes = np.expand_dims(axes, axis=0)

eval_rows = []
for r, (name, rgb_i) in enumerate(loaded_images):
    classic_vis, classic_mask, _, classic_xyxy, classic_strat, classic_ms = classic_nb1_detect(rgb_i)
    ssd_dets, ssd_ms = run_ssd_detect(rgb_i)
    ssd_vis = draw_ssd_boxes(rgb_i, ssd_dets)
    gc_mask = grabcut_from_boxes(rgb_i, ssd_dets)
    gc_overlay = make_overlay(ssd_vis, gc_mask, color=(255, 70, 20), alpha=0.35)

    ssd_top_box = ssd_dets[0]["xyxy_px"] if ssd_dets else None
    iou_nb1_vs_ssd = compute_iou_xyxy(classic_xyxy, ssd_top_box)
    ssd_summary = ", ".join(f"{d['class']}({d['score']:.2f})" for d in ssd_dets) or "Tidak terdeteksi"
    speedup = classic_ms / max(0.1, ssd_ms)

    eval_rows.append({
        "Gambar": name[:24],
        "Strategi_NB1": classic_strat,
        "NB1_Waktu_ms": round(classic_ms, 1),
        "SSD_Prediksi": ssd_summary,
        "SSD_Waktu_ms": round(ssd_ms, 1),
        "IoU_NB1_vs_SSD": round(iou_nb1_vs_ssd, 3),
        "SSD_Lebih_Cepat": f"{speedup:.1f}x",
    })

    axes[r, 0].imshow(rgb_i)
    axes[r, 0].set_title(f"Input: {name[:24]}", fontsize=9.2)
    axes[r, 1].imshow(classic_vis)
    axes[r, 1].set_title(f"PCV Klasik NB1 ({classic_strat})\nIoU={iou_nb1_vs_ssd:.2f} | {classic_ms:.0f} ms (Tanpa Label)", fontsize=9)
    axes[r, 2].imshow(ssd_vis)
    axes[r, 2].set_title(f"FERQUEUE SSD MobileNetV2\n{ssd_summary} | {ssd_ms:.0f} ms", fontsize=9)
    axes[r, 3].imshow(gc_overlay)
    axes[r, 3].set_title("FERQUEUE + GrabCut ROI\n(Segmentasi Pasca-Deteksi)", fontsize=9)
    for c in range(4):
        axes[r, c].axis("off")

fig.suptitle("Perbandingan Hasil Akhir: PCV Klasik Adaptif (NB1) vs SSD MobileNetV2 (FERQUEUE)", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
save_fig(fig, "04_banding_hasil_akhir_side_by_side.png")
plt.show()

df_eval = pd.DataFrame(eval_rows)
df_eval.to_csv(OUT_DIR / "04_tabel_hasil_uji.csv", index=False)
display(df_eval)

# Grafik batang waktu eksekusi: terlihat jelas SSD jauh lebih cepat utk kamera live!
fig_t, ax_t = plt.subplots(figsize=(max(8.0, 2.6 * n_imgs), 4.5))
x_pos = np.arange(n_imgs)
width_b = 0.36
b1 = ax_t.bar(x_pos - width_b / 2, df_eval["NB1_Waktu_ms"], width_b, label="PCV Klasik Adaptif NB1 (Saliency + GrabCut)", color="#4c72b0")
b2 = ax_t.bar(x_pos + width_b / 2, df_eval["SSD_Waktu_ms"], width_b, label="FERQUEUE SSD MobileNetV2 (Produksi)", color="#2e8b57")
max_h = max(df_eval["NB1_Waktu_ms"].max(), df_eval["SSD_Waktu_ms"].max())
for bar in list(b1) + list(b2):
    val = bar.get_height()
    ax_t.text(bar.get_x() + bar.get_width() / 2, val + max_h * 0.02, f"{val:.0f}ms", ha="center", fontsize=8.5, fontweight="bold")
ax_t.set_xticks(x_pos)
ax_t.set_xticklabels([g[:20] for g in df_eval["Gambar"]], rotation=10, ha="right")
ax_t.set_ylim(0, max_h * 1.22)
ax_t.set_ylabel("Waktu Eksekusi per Gambar (ms)")
ax_t.set_title("Perbandingan Kecepatan Inferensi: PCV Klasik Adaptif vs SSD MobileNetV2")
ax_t.legend(loc="upper right")
ax_t.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_fig(fig_t, "05_grafik_waktu_eksekusi.png")
plt.show()

## Cell 8 — Matriks Perbandingan Komprehensif (Dipakai vs PCV Klasik NB1 vs Metode Tidak Digunakan) & Unduh ZIP PPT

Tabel ini merangkum seluruh temuan kuantitatif — termasuk hasil uji kalibrasi **1.079 gambar `data/inference`** — menjadi 1 matriks keputusan (CSV + PNG 300 DPI) yang siap langsung ditempel di slide kesimpulan PowerPoint.

In [ ]:
comparison_matrix = [
    {
        "Metode / Arsitektur": "SSD MobileNetV2 FPNLite\n(DIPAKAI — FERQUEUE)",
        "Kategori": "Deep Learning\n(Single-Stage)",
        "Segmentasi": "Tidak wajib di awal\n(GrabCut opsional review)",
        "Ekstraksi Fitur": "Otomatis CNN\n(MobileNetV2 + FPN P3-P7)",
        "Output & Kecepatan": "Bbox + Label + Skor\n(~25-45 ms/frame)",
        "Ketahanan Kondisi\nNyata": "Tinggi (tahan warna, silau,\nframe kosong, multi-objek)",
        "Alasan Dipakai / Tidak Dipakai": "DIPILIH: Ringan (~10 MB), real-time di CPU\nlokal & Flask, cocok anotasi bbox YOLO.",
    },
    {
        "Metode / Arsitektur": "Saliency + GrabCut if-else\n+ Geometri/HOG (NB1)",
        "Kategori": "PCV Klasik Adaptif\n(Rule-Based)",
        "Segmentasi": "Saliency (Lab+FFT)\n+ GrabCut adaptif",
        "Ekstraksi Fitur": "Manual (Geometri\nKontur + HOG + GLCM)",
        "Output & Kecepatan": "Bbox tanpa label\n(~980 ms/gambar)",
        "Ketahanan Kondisi\nNyata": "Baik (87.7% IoU>=0.5 di\n1.079 img; silau 37.2%)",
        "Alasan Dipakai / Tidak Dipakai": "PEMBANDING NB1: Transparan utk materi PCV,\ntetapi ~25-30x lebih lambat & tanpa label.",
    },
    {
        "Metode / Arsitektur": "Otsu / Canny Naive /\nK-Means / Watershed / HSV",
        "Kategori": "PCV Klasik Dasar\n(Tidak Dipakai)",
        "Segmentasi": "Threshold/Tepi/Klaster\ntanpa panduan objek",
        "Ekstraksi Fitur": "Intensitas / Tepi /\nWarna piksel mentah",
        "Output & Kecepatan": "Mask biner\n(~10-25 ms/gambar)",
        "Ketahanan Kondisi\nNyata": "Rendah (Canny bocor\nfull-frame 57.7% di 1.079)",
        "Alasan Dipakai / Tidak Dipakai": "TIDAK DIPAKAI utk kendaraan: bodi terpecah\n& bocor latar (Otsu hanya utk kalibrasi koin).",
    },
    {
        "Metode / Arsitektur": "HOG + SVM /\nHaar Cascade",
        "Kategori": "ML Klasik\n(Tidak Dipakai)",
        "Segmentasi": "Sliding window\nmulti-skala",
        "Ekstraksi Fitur": "Manual (HOG / fitur\nHaar persegi)",
        "Output & Kecepatan": "Bbox + Label\n(terbatas)",
        "Ketahanan Kondisi\nNyata": "Sedang (kaku terhadap\nsudut pandang/oklusi)",
        "Alasan Dipakai / Tidak Dipakai": "TIDAK DIPAKAI: Akurasi kalah dari CNN &\nsliding window lambat pada multi-skala.",
    },
    {
        "Metode / Arsitektur": "Faster R-CNN /\nSSD ResNet50",
        "Kategori": "Deep Learning\n(Two-Stage / Berat)",
        "Segmentasi": "Region Proposal\nNetwork (RPN)",
        "Ekstraksi Fitur": "Otomatis CNN\n(ResNet/VGG berat)",
        "Output & Kecepatan": "Bbox + Label + Skor\n(berat di CPU)",
        "Ketahanan Kondisi\nNyata": "Sangat Tinggi",
        "Alasan Dipakai / Tidak Dipakai": "TIDAK DIPAKAI: Bobot besar & terlalu lambat\nuntuk CPU laptop tanpa GPU NVIDIA wajib.",
    },
    {
        "Metode / Arsitektur": "Mask R-CNN / U-Net\n(Instance/Semantic Seg.)",
        "Kategori": "Deep Learning\n(Segmentasi Piksel)",
        "Segmentasi": "Eksplisit per piksel\n(End-to-End Mask)",
        "Ekstraksi Fitur": "Otomatis CNN\n(Backbone + Mask Head)",
        "Output & Kecepatan": "Mask Piksel + Bbox\n+ Label (berat)",
        "Ketahanan Kondisi\nNyata": "Sangat Tinggi",
        "Alasan Dipakai / Tidak Dipakai": "TIDAK DIPAKAI: Butuh anotasi poligon per\npiksel (dataset YOLO hanya bbox 5 kolom).",
    },
]

df_matrix = pd.DataFrame(comparison_matrix)
df_matrix.to_csv(OUT_DIR / "06_matriks_perbandingan_metode.csv", index=False)
display(df_matrix)

fig_tbl, ax_tbl = plt.subplots(figsize=(19.0, 7.0))
ax_tbl.axis("off")
tbl = ax_tbl.table(
    cellText=df_matrix.values,
    colLabels=df_matrix.columns,
    cellLoc="left",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.3)
tbl.scale(1.0, 2.65)

for (row_i, col_i), cell in tbl.get_celld().items():
    cell.set_edgecolor("#cccccc")
    if row_i == 0:
        cell.set_facecolor("#1f3a5f")
        cell.set_text_props(color="white", weight="bold")
    elif row_i == 1:
        cell.set_facecolor("#e6f4ea")
        cell.set_text_props(weight="bold" if col_i == 0 else "normal")
    elif row_i == 2:
        cell.set_facecolor("#eef4fb")
        cell.set_text_props(weight="bold" if col_i == 0 else "normal")
    else:
        cell.set_facecolor("#ffffff" if row_i % 2 == 1 else "#f9f9f9")

ax_tbl.set_title(
    "Matriks Perbandingan Metode: FERQUEUE (Dipakai) vs PCV Klasik Adaptif (NB1) vs Metode yang Tidak Digunakan",
    fontsize=13.5, fontweight="bold", pad=12,
)
plt.tight_layout()
save_fig(fig_tbl, "06_matriks_perbandingan_metode.png")
plt.show()

zip_path = OUT_DIR.parent / "nb3_perbandingan_metode_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print(f"\nSemua gambar & tabel PPT tersimpan di: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")
# Hapus tanda '#' di bawah jika ingin langsung mengunduh ZIP di Colab:
# if IN_COLAB:
#     colab_files.download(str(zip_path))